In [1]:
from pathlib import Path
import pandas as pd

# Ruta del archivo generado por el segundo script
CARPETA = Path(r"C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan")
ARCHIVO = CARPETA / "base_dashboard_rotacion.xlsx"
HOJA = "dashboard_ventas_canal"

# Cargar la hoja final
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# Validaciones mínimas
for col in ["canal", "venta_total"]:
    if col not in df.columns:
        raise KeyError(f"No encontré la columna '{col}' en la hoja {HOJA}")

# Normalizar numéricos
for col in ["unidades", "venta_total", "pedidos"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Ventas totales por canal
resumen_canal = (
    df.groupby("canal", as_index=False)
      .agg(
          ventas_totales_monto=("venta_total", "sum"),
          unidades=("unidades", "sum") if "unidades" in df.columns else ("venta_total", "size"),
          pedidos=("pedidos", "sum") if "pedidos" in df.columns else ("venta_total", "size"),
      )
      .sort_values("ventas_totales_monto", ascending=False)
      .reset_index(drop=True)
)

print(resumen_canal.to_string(index=False))

# Opcional: guardar el resumen en un nuevo Excel
salida = CARPETA / "ventas_totales_por_canal_3m.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen_canal.to_excel(writer, sheet_name="ventas_por_canal_3m", index=False)

print(f"\nArchivo generado: {salida}")

        canal  ventas_totales_monto  unidades  pedidos
         Full           48106720.68     40601    38110
         Drop            3745593.08      1970     1889
    LIVERPOOL             407513.30       189      187
       AMAZON             356587.24       225      210
MERCADO LIBRE             144787.25        66       65
      WALMART             133224.00        41       39
      ELEKTRA              73716.00        32       31
       COPPEL               4311.00         9        9
      TIK TOK                359.00         1        1

Archivo generado: C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan\ventas_totales_por_canal_3m.xlsx
